# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [5]:
!pip install -q chromadb peft accelerate transformers sentence-transformers
!pip install -q --upgrade "torchao>=0.16.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [6]:
# YOUR CODE HERE
import os
import shutil
import tarfile
import zipfile

lora_dir = "./intent_lora_best"
chroma_dir = "./chroma_db"
os.makedirs(lora_dir, exist_ok=True)
os.makedirs(chroma_dir, exist_ok=True)


def inspect_file(path):
  if not os.path.exists(path):
    print(f"[MISSING] {path}")
    return
  size = os.path.getsize(path)
  with open(path, "rb") as f:
    header = f.read(16)
  print(f"[FILE] {path} | {size} bytes ({size/(1024*1024):.2f} MB) | header={header!r}")


inspect_file("lora_training_artifacts.zip")
inspect_file("chroma_db.zip")


def extract_archive(archive_path, extract_to):
  os.makedirs(extract_to, exist_ok=True)
  if zipfile.is_zipfile(archive_path):
    with zipfile.ZipFile(archive_path, "r") as zf:
      zf.extractall(extract_to)
    print(f"Extracted {archive_path} as ZIP -> {extract_to}")
    return True
  if tarfile.is_tarfile(archive_path):
    with tarfile.open(archive_path, "r:*") as tf:
      tf.extractall(extract_to)
    print(f"Extracted {archive_path} as TAR -> {extract_to}")
    return True
  rc = os.system(f"unzip -oq '{archive_path}' -d '{extract_to}'")
  if rc == 0:
    print(f"Extracted {archive_path} via system unzip -> {extract_to}")
    return True
  print(f"[ERROR] {archive_path} is neither a valid ZIP nor TAR archive (likely truncated/corrupted).")
  return False


def copy_target_folder(root_dir, target_name, dest_dir):
  if not os.path.exists(root_dir):
    return False
  for dirpath, dirnames, _ in os.walk(root_dir):
    if os.path.basename(dirpath) == target_name:
      shutil.copytree(dirpath, dest_dir, dirs_exist_ok=True)
      print(f"Copied '{dirpath}' -> '{dest_dir}'")
      return True
  # Fallback: if extracted files are directly inside extract_to without a wrapper folder
  if os.listdir(root_dir):
    shutil.copytree(root_dir, dest_dir, dirs_exist_ok=True)
    print(f"Copied extracted contents from '{root_dir}' -> '{dest_dir}'")
    return True
  return False


lora_ok = extract_archive("lora_training_artifacts.zip", "./lora_extracted")
if lora_ok:
  copy_target_folder("./lora_extracted", "intent_lora_best", lora_dir)

chroma_ok = extract_archive("chroma_db.zip", "./chroma_extracted")
if chroma_ok:
  copy_target_folder("./chroma_extracted", "chroma_db", chroma_dir)

print("\nContents of ./intent_lora_best:", os.listdir(lora_dir))
print("Contents of ./chroma_db:", os.listdir(chroma_dir))

[FILE] lora_training_artifacts.zip | 16137950 bytes (15.39 MB) | header=b'PK\x03\x04\x14\x00\x00\x00\x08\x00\x03ax]\xa9\xf8'
[FILE] chroma_db.zip | 60433 bytes (0.06 MB) | header=b'PK\x03\x04\n\x00\x00\x00\x00\x00\xb2yw]\x00\x00'
Extracted lora_training_artifacts.zip as ZIP -> ./lora_extracted
Copied './lora_extracted/intent_lora_best' -> './intent_lora_best'
Extracted chroma_db.zip as ZIP -> ./chroma_extracted
Copied './chroma_extracted/chroma_db' -> './chroma_db'

Contents of ./intent_lora_best: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']
Contents of ./chroma_db: ['acdb434e-4e81-4c8a-95f1-07b75df419dc', 'chroma.sqlite3']


In [7]:
# YOUR CODE HERE
import json
import os
import re
import chromadb
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---------------------------------------------------------
# 1. Load Base Model + Merge LoRA Adapters (./intent_lora_best)
# ---------------------------------------------------------
lora_dir = "./intent_lora_best"
adapter_cfg_path = os.path.join(lora_dir, "adapter_config.json")
with open(adapter_cfg_path, "r") as f:
  adapter_cfg = json.load(f)

# Use the base model recorded in adapter_config.json, or set explicitly if needed
base_model_name = adapter_cfg.get(
    "base_model_name_or_path", "Qwen/Qwen2.5-1.5B-Instruct"
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Merge LoRA weights into the base model for faster inference
router_model = PeftModel.from_pretrained(base_model, lora_dir).merge_and_unload()
router_model.eval()
print(f"Loaded and merged LoRA from {lora_dir} onto {base_model_name}")

# ---------------------------------------------------------
# 2. Intent Router (JSON-only with 4 few-shot examples + regex fallback)
# ---------------------------------------------------------
SYSTEM_PROMPT = """You are a strict customer-support intent router.
Classify the user message into a concise snake_case intent and output ONLY valid JSON: {"intent": "<intent_name>"}.
Do not add explanations or markdown fences."""

FEW_SHOT_MESSAGES = [
    {"role": "user", "content": "Where is my package? Tracking number 88412."},
    {"role": "assistant", "content": '{"intent": "track_order"}'},
    {"role": "user", "content": "I want my money back because the item arrived damaged."},
    {"role": "assistant", "content": '{"intent": "refund_request"}'},
    {"role": "user", "content": "How do I change my shipping address before it ships?"},
    {"role": "assistant", "content": '{"intent": "update_address"}'},
    {"role": "user", "content": "Can I cancel the subscription I started yesterday?"},
    {"role": "assistant", "content": '{"intent": "cancel_subscription"}'},
]


def route_intent(query: str) -> dict:
  messages = (
      [{"role": "system", "content": SYSTEM_PROMPT}]
      + FEW_SHOT_MESSAGES
      + [{"role": "user", "content": query}]
  )
  if hasattr(tokenizer, "apply_chat_template"):
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
  else:
    prompt_text = (
        SYSTEM_PROMPT
        + "\n"
        + "\n".join(f"{m['role']}: {m['content']}" for m in FEW_SHOT_MESSAGES)
        + f"\nuser: {query}\nassistant: "
    )

  inputs = tokenizer(prompt_text, return_tensors="pt").to(router_model.device)
  with torch.no_grad():
    outputs = router_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )

  gen_tokens = outputs[0][inputs["input_ids"].shape[1] :]
  raw_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

  # Safety net for stray text around JSON
  match = re.search(r"\{.*?\}", raw_text, flags=re.DOTALL)
  if match:
    try:
      parsed = json.loads(match.group(0))
      if isinstance(parsed, dict) and "intent" in parsed:
        return {
            "intent": str(parsed["intent"]).strip(),
            "raw": raw_text,
            "parsed_ok": True,
        }
    except json.JSONDecodeError:
      pass

  return {"intent": "unknown", "raw": raw_text, "parsed_ok": False}


# Map structured intent -> SOP header search string
INTENT_TO_SOP_QUERY = {
    "track_order": "# Track Order",
    "order_tracking": "# Track Order",
    "refund_request": "# Refund Policy",
    "return_refund": "# Refund Policy",
    "cancel_order": "# Cancel Order",
    "cancel_subscription": "# Subscription Cancellation",
    "update_address": "# Shipping Address",
    "shipping_issue": "# Shipping",
    "billing_issue": "# Billing",
    "account_help": "# Account Support",
}


def intent_to_search_string(intent: str, fallback_query: str, parsed_ok: bool) -> str:
  if not parsed_ok or not intent or intent == "unknown":
    return fallback_query
  normalized = intent.lower().strip().replace(" ", "_")
  if normalized in INTENT_TO_SOP_QUERY:
    return INTENT_TO_SOP_QUERY[normalized]
  # Generic SOP header conversion from snake_case intent
  return "# " + " ".join(word.capitalize() for word in normalized.split("_"))


# ---------------------------------------------------------
# 3. Connect to Chroma DB (./chroma_db) & Retrieve SOP Context
# ---------------------------------------------------------
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collections = chroma_client.list_collections()
assert collections, "No collections found in ./chroma_db"
collection = collections[0]
print(f"Connected to Chroma collection: '{collection.name}' (count={collection.count()})")


def retrieve_sop(search_query: str, k: int = 3) -> list[str]:
  results = collection.query(query_texts=[search_query], n_results=k)
  docs = results.get("documents", [[]])[0]
  return docs if docs else []


# ---------------------------------------------------------
# 4. Generate Grounded Answer + End-to-End Validation
# ---------------------------------------------------------
def generate_answer(query: str, sop_chunks: list[str]) -> str:
  context_block = (
      "\n\n---\n\n".join(sop_chunks)
      if sop_chunks
      else "No matching SOP document found."
  )
  rag_prompt = (
      "Answer the customer's question accurately using ONLY the provided SOP context.\n"
      "If the SOP does not cover part of the question, state what is supported.\n\n"
      f"SOP Context:\n{context_block}\n\n"
      f"Customer Query: {query}\n"
      "Answer:"
  )
  inputs = tokenizer(rag_prompt, return_tensors="pt").to(router_model.device)
  with torch.no_grad():
    outputs = router_model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
  gen_tokens = outputs[0][inputs["input_ids"].shape[1] :]
  return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


def hybrid_rag_pipeline(query: str, k: int = 3) -> dict:
  route_info = route_intent(query)
  search_str = intent_to_search_string(
      intent=route_info["intent"],
      fallback_query=query,
      parsed_ok=route_info["parsed_ok"],
  )
  sop_chunks = retrieve_sop(search_str, k=k)
  answer = generate_answer(query, sop_chunks)
  return {
      "query": query,
      "intent": route_info["intent"],
      "raw_router_output": route_info["raw"],
      "search_string": search_str,
      "retrieved_sop": sop_chunks,
      "answer": answer,
  }


# Validate end-to-end on test_query
test_query = "I've been waiting 10 days for my order and tracking hasn't moved—where is it?"
result = hybrid_rag_pipeline(test_query, k=3)

print("\n=== END-TO-END VALIDATION ===")
print("Query:        ", result["query"])
print("Intent:       ", result["intent"], f"(raw: {result['raw_router_output']})")
print("Search String:", result["search_string"])
print("SOP Snippets: ", len(result["retrieved_sop"]))
if result["retrieved_sop"]:
  print("Top SOP Preview:\n", result["retrieved_sop"][0][:400])
print("\nFinal Answer:\n", result["answer"])

W1124 14:39:14.648000 398 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1124 14:39:14.690000 398 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (4 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded and merged LoRA from ./intent_lora_best onto Qwen/Qwen2.5-1.5B-Instruct
Connected to Chroma collection: 'langchain' (count=5)


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:28<00:00, 2.96MiB/s]



=== END-TO-END VALIDATION ===
Query:         I've been waiting 10 days for my order and tracking hasn't moved—where is it?
Intent:        check_delivery_status (raw: {"intent": "check_delivery_status"})
Search String: # Check Delivery Status
SOP Snippets:  3
Top SOP Preview:
 # Shipping Delays

## Scope
This procedure covers orders that have shipped but are running behind the
estimated delivery date, as well as orders stuck in a pre-shipment state.

## Standard Timelines
Domestic orders deliver within 3–7 business days; international orders within
10–21 business days. An order is only considered "delayed" once it has passed
the upper bound of its quoted window. Carrier

Final Answer:
 The order was delayed due to unforeseen circumstances, such as bad weather or customs clearance issues. We're currently working on finding you a new shipment. Please allow us about 24-48 hours for your updated information. In the meantime, we'll keep you informed of our progress. Thank you for your patie

### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [8]:
import difflib
import json
import os
import re
import subprocess
import sys
import pandas as pd

# Ensure evaluation libraries are installed
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "rouge-score",
    "nltk",
    "tqdm",
])

from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
import torch
from tqdm.auto import tqdm

# ---------------------------------------------------------
# 0. Guard 4.3.1 Globals & Intent-to-SOP Mapping After Runtime Reconnect
# ---------------------------------------------------------
if "INTENT_TO_SOP_QUERY" not in globals():
  INTENT_TO_SOP_QUERY = {
      "track_order": "# Track Order",
      "order_tracking": "# Track Order",
      "refund_request": "# Refund Policy",
      "return_refund": "# Refund Policy",
      "cancel_order": "# Cancel Order",
      "cancel_subscription": "# Subscription Cancellation",
      "update_address": "# Shipping Address",
      "shipping_issue": "# Shipping",
      "billing_issue": "# Billing",
      "account_help": "# Account Support",
  }

INTENT_TO_SOP_QUERY["check_delivery_status"] = "# Shipping Delays"

if "intent_to_search_string" not in globals():

  def intent_to_search_string(
      intent: str, fallback_query: str, parsed_ok: bool
  ) -> str:
    if not parsed_ok or not intent or intent == "unknown":
      return fallback_query
    normalized = intent.lower().strip().replace(" ", "_")
    if normalized in INTENT_TO_SOP_QUERY:
      return INTENT_TO_SOP_QUERY[normalized]
    return "# " + " ".join(word.capitalize() for word in normalized.split("_"))


if (
    "router_model" not in globals()
    or "tokenizer" not in globals()
    or "collection" not in globals()
):
  import chromadb
  from peft import PeftModel
  from transformers import AutoModelForCausalLM, AutoTokenizer

  lora_dir = "./intent_lora_best"
  adapter_cfg_path = os.path.join(lora_dir, "adapter_config.json")
  with open(adapter_cfg_path, "r") as f:
    adapter_cfg = json.load(f)

  base_model_name = adapter_cfg.get(
      "base_model_name_or_path", "Qwen/Qwen2.5-1.5B-Instruct"
  )
  tokenizer = AutoTokenizer.from_pretrained(
      base_model_name, trust_remote_code=True
  )
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  base_model = AutoModelForCausalLM.from_pretrained(
      base_model_name,
      dtype=torch.float16,
      device_map="auto",
      trust_remote_code=True,
  )
  router_model = PeftModel.from_pretrained(
      base_model, lora_dir
  ).merge_and_unload()
  router_model.eval()

  chroma_client = chromadb.PersistentClient(path="./chroma_db")
  collections = chroma_client.list_collections()
  assert collections, "No collections found in ./chroma_db"
  collection = collections[0]

if "route_intent" not in globals():
  SYSTEM_PROMPT = """You are a strict customer-support intent router.
Classify the user message into a concise snake_case intent and output ONLY valid JSON: {"intent": "<intent_name>"}.
Do not add explanations or markdown fences."""

  FEW_SHOT_MESSAGES = [
      {
          "role": "user",
          "content": "Where is my package? Tracking number 88412.",
      },
      {"role": "assistant", "content": '{"intent": "track_order"}'},
      {
          "role": "user",
          "content": "I want my money back because the item arrived damaged.",
      },
      {"role": "assistant", "content": '{"intent": "refund_request"}'},
      {
          "role": "user",
          "content": "How do I change my shipping address before it ships?",
      },
      {"role": "assistant", "content": '{"intent": "update_address"}'},
      {
          "role": "user",
          "content": "Can I cancel the subscription I started yesterday?",
      },
      {"role": "assistant", "content": '{"intent": "cancel_subscription"}'},
  ]

  def route_intent(query: str) -> dict:
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + FEW_SHOT_MESSAGES
        + [{"role": "user", "content": query}]
    )
    if hasattr(tokenizer, "apply_chat_template"):
      prompt_text = tokenizer.apply_chat_template(
          messages, tokenize=False, add_generation_prompt=True
      )
    else:
      prompt_text = (
          SYSTEM_PROMPT
          + "\n"
          + "\n".join(
              f"{m['role']}: {m['content']}" for m in FEW_SHOT_MESSAGES
          )
          + f"\nuser: {query}\nassistant: "
      )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(router_model.device)
    with torch.no_grad():
      outputs = router_model.generate(
          **inputs,
          max_new_tokens=30,
          do_sample=False,
          pad_token_id=tokenizer.pad_token_id,
      )
    gen_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    raw_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

    match = re.search(r"\{.*?\}", raw_text, flags=re.DOTALL)
    if match:
      try:
        parsed = json.loads(match.group(0))
        if isinstance(parsed, dict) and "intent" in parsed:
          return {
              "intent": str(parsed["intent"]).strip(),
              "raw": raw_text,
              "parsed_ok": True,
          }
      except json.JSONDecodeError:
        pass
    return {"intent": "unknown", "raw": raw_text, "parsed_ok": False}


if "retrieve_sop" not in globals():

  def retrieve_sop(search_query: str, k: int = 3) -> list[str]:
    results = collection.query(query_texts=[search_query], n_results=k)
    docs = results.get("documents", [[]])[0]
    return docs if docs else []


if "generate_answer" not in globals():

  def generate_answer(query: str, sop_chunks: list[str]) -> str:
    context_block = (
        "\n\n---\n\n".join(sop_chunks)
        if sop_chunks
        else "No matching SOP document found."
    )
    rag_prompt = (
        "Answer the customer's question accurately using ONLY the provided SOP"
        " context.\n"
        "If the SOP does not cover part of the question, state what is"
        " supported.\n\n"
        f"SOP Context:\n{context_block}\n\n"
        f"Customer Query: {query}\n"
        "Answer:"
    )
    inputs = tokenizer(rag_prompt, return_tensors="pt").to(router_model.device)
    with torch.no_grad():
      outputs = router_model.generate(
          **inputs,
          max_new_tokens=180,
          do_sample=False,
          pad_token_id=tokenizer.pad_token_id,
      )
    gen_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


if "hybrid_rag_pipeline" not in globals():

  def hybrid_rag_pipeline(query: str, k: int = 3) -> dict:
    route_info = route_intent(query)
    search_str = intent_to_search_string(
        intent=route_info["intent"],
        fallback_query=query,
        parsed_ok=route_info["parsed_ok"],
    )
    sop_chunks = retrieve_sop(search_str, k=k)
    answer = generate_answer(query, sop_chunks)
    return {
        "query": query,
        "intent": route_info["intent"],
        "raw_router_output": route_info["raw"],
        "search_string": search_str,
        "retrieved_sop": sop_chunks,
        "answer": answer,
    }


# ---------------------------------------------------------
# 1. Load Held-Out Test Set (df_test.csv) & Reference Outputs (outputs.json)
# ---------------------------------------------------------
df_test = pd.read_csv("df_test.csv")
print("df_test shape:", df_test.shape)
print("df_test columns:", list(df_test.columns))

query_col = next(
    (
        c
        for c in df_test.columns
        if c.lower()
        in ["query", "text", "input", "instruction", "utterance", "prompt"]
    ),
    df_test.columns[0],
)
intent_col = next(
    (
        c
        for c in df_test.columns
        if c.lower()
        in ["intent", "label", "target", "category", "output", "ground_truth"]
    ),
    df_test.columns[1],
)
print(f"Using query_col='{query_col}', intent_col='{intent_col}'")


def extract_clean_intent(val) -> str:
  s = str(val).strip()
  match = re.search(r"\{.*?\}", s, flags=re.DOTALL)
  if match:
    try:
      obj = json.loads(match.group(0))
      if isinstance(obj, dict) and "intent" in obj:
        return str(obj["intent"]).strip().lower()
    except json.JSONDecodeError:
      pass
  return s.strip('"').strip("'").strip().lower()


df_test["clean_true_intent"] = df_test[intent_col].apply(extract_clean_intent)

# ---------------------------------------------------------
# 2. Identify Regex-Filtered Adversarial Subset
# ---------------------------------------------------------
ADVERSARIAL_REGEX = re.compile(
    r"(\bbut\b|\bhowever\b|\binstead\b|\balso\b|\bnot\b|\bnever\b|\bdon't\b|\bwon't\b|\bcancel\b.*\brefund\b|\brefund\b.*\bcancel\b|\burgent\b|\basap\b|\bwtf\b|\bangry\b|\bwrong\b|\bmissing\b|\?\s*\w+)",
    flags=re.IGNORECASE,
)
df_test["is_adversarial"] = (
    df_test[query_col]
    .astype(str)
    .apply(lambda x: bool(ADVERSARIAL_REGEX.search(x)))
)
print(
    f"Adversarial subset count: {df_test['is_adversarial'].sum()} /"
    f" {len(df_test)}"
)


# ---------------------------------------------------------
# 3. Run Fine-Tuned Router on Held-Out Test Split
# ---------------------------------------------------------
def evaluate_router_on_df(df: pd.DataFrame) -> pd.DataFrame:
  preds, raw_outs, parsed_flags = [], [], []
  for q in tqdm(df[query_col].astype(str).tolist(), desc="Routing Test Queries"):
    res = route_intent(q)
    preds.append(res["intent"].strip().lower())
    raw_outs.append(res["raw"])
    parsed_flags.append(bool(res["parsed_ok"]))

  out_df = df.copy()
  out_df["pred_intent"] = preds
  out_df["raw_router_output"] = raw_outs
  out_df["format_adherence"] = parsed_flags
  out_df["exact_match"] = out_df["pred_intent"] == out_df["clean_true_intent"]
  out_df["fuzzy_score"] = [
      difflib.SequenceMatcher(None, p, t).ratio()
      for p, t in zip(out_df["pred_intent"], out_df["clean_true_intent"])
  ]
  out_df["fuzzy_match"] = out_df["fuzzy_score"] >= 0.80
  return out_df


eval_df = evaluate_router_on_df(df_test)


def summarize_router_metrics(df_slice: pd.DataFrame, split_name: str) -> dict:
  return {
      "Split": split_name,
      "N": len(df_slice),
      "Format_Adherence_%": round(df_slice["format_adherence"].mean() * 100, 2),
      "Exact_Match_%": round(df_slice["exact_match"].mean() * 100, 2),
      "Fuzzy_Match_%": round(df_slice["fuzzy_match"].mean() * 100, 2),
      "Avg_Fuzzy_Similarity": round(df_slice["fuzzy_score"].mean(), 4),
  }


adv_df = eval_df[eval_df["is_adversarial"]]
router_summary = pd.DataFrame([
    summarize_router_metrics(eval_df, "Held-Out Test (All)"),
    summarize_router_metrics(
        adv_df if len(adv_df) > 0 else eval_df, "Adversarial Subset (Regex)"
    ),
])

print("\n=== ROUTER EVALUATION (V2 FINE-TUNED) ===")
print(router_summary.to_string(index=False))

# ---------------------------------------------------------
# 4. Evaluate End-to-End Synthesis (ROUGE & BLEU vs outputs.json)
# ---------------------------------------------------------
ref_pairs = []
if os.path.exists("outputs.json"):
  with open("outputs.json", "r") as f:
    ref_data = json.load(f)

  if isinstance(ref_data, list):
    for item in ref_data:
      if isinstance(item, dict):
        q = item.get("query") or item.get("question") or item.get("input")
        ref = (
            item.get("reference")
            or item.get("ground_truth")
            or item.get("expected_answer")
            or item.get("answer")
            or item.get("output")
        )
        if q and ref:
          ref_pairs.append((str(q), str(ref)))
  elif isinstance(ref_data, dict):
    for k, v in ref_data.items():
      if isinstance(v, str):
        ref_pairs.append((str(k), v))
      elif isinstance(v, dict):
        q = v.get("query") or str(k)
        ref = (
            v.get("reference")
            or v.get("ground_truth")
            or v.get("answer")
            or v.get("output")
        )
        if ref:
          ref_pairs.append((str(q), str(ref)))

MAX_SYNTH_SAMPLES = min(len(ref_pairs), 50)
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth_fn = SmoothingFunction().method1

r1_scores, r2_scores, rl_scores, bleu_scores = [], [], [], []
synth_records = []

for q, ref_ans in tqdm(
    ref_pairs[:MAX_SYNTH_SAMPLES], desc="Evaluating RAG Synthesis (ROUGE/BLEU)"
):
  pipe_out = hybrid_rag_pipeline(q, k=3)
  pred_ans = pipe_out["answer"]

  scores = scorer.score(ref_ans, pred_ans)
  r1 = scores["rouge1"].fmeasure
  r2 = scores["rouge2"].fmeasure
  rl = scores["rougeL"].fmeasure

  ref_tokens = ref_ans.lower().split()
  pred_tokens = pred_ans.lower().split()
  bleu = (
      sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth_fn)
      if pred_tokens
      else 0.0
  )

  r1_scores.append(r1)
  r2_scores.append(r2)
  rl_scores.append(rl)
  bleu_scores.append(bleu)
  synth_records.append({
      "query": q,
      "routed_intent": pipe_out["intent"],
      "search_string": pipe_out["search_string"],
      "reference_answer": ref_ans,
      "v2_answer": pred_ans,
      "rouge1": round(r1, 4),
      "rouge2": round(r2, 4),
      "rougeL": round(rl, 4),
      "bleu": round(bleu, 4),
  })

synth_df = pd.DataFrame(synth_records)
v2_synth_summary = pd.DataFrame([{
    "Samples": len(synth_df),
    "ROUGE-1": round(sum(r1_scores) / max(len(r1_scores), 1), 4),
    "ROUGE-2": round(sum(r2_scores) / max(len(r2_scores), 1), 4),
    "ROUGE-L": round(sum(rl_scores) / max(len(rl_scores), 1), 4),
    "BLEU": round(sum(bleu_scores) / max(len(bleu_scores), 1), 4),
}])

print("\n=== END-TO-END SYNTHESIS METRICS (V2) ===")
print(v2_synth_summary.to_string(index=False))

eval_df.to_csv("v2_router_eval.csv", index=False)
synth_df.to_csv("v2_synthesis_eval.csv", index=False)

df_test shape: (391, 4)
df_test columns: ['instruction', 'text', 'intent', 'category']
Using query_col='instruction', intent_col='intent'
Adversarial subset count: 37 / 391


Routing Test Queries:   0%|          | 0/391 [00:00<?, ?it/s]


=== ROUTER EVALUATION (V2 FINE-TUNED) ===
                     Split   N  Format_Adherence_%  Exact_Match_%  Fuzzy_Match_%  Avg_Fuzzy_Similarity
       Held-Out Test (All) 391               100.0          32.23          42.46                0.7438
Adversarial Subset (Regex)  37               100.0          29.73          43.24                0.7160


Evaluating RAG Synthesis (ROUGE/BLEU):   0%|          | 0/6 [00:00<?, ?it/s]


=== END-TO-END SYNTHESIS METRICS (V2) ===
 Samples  ROUGE-1  ROUGE-2  ROUGE-L   BLEU
       6   0.1022   0.0066    0.059 0.0027


#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [9]:
# YOUR CODE HERE
import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Load V1 Metrics (v1_metrics.csv) & Gather V2 Metrics
# ---------------------------------------------------------
assert os.path.exists("v1_metrics.csv"), "v1_metrics.csv not found in working directory!"
v1_df = pd.read_csv("v1_metrics.csv")
print("=== RAW v1_metrics.csv ===")
print(v1_df.to_string(index=False))

# Build a unified dictionary of V2 metrics from Task 4.4.1 variables (or fallback CSVs)
v2_metrics_map = {}

if "router_summary" in globals() and isinstance(router_summary, pd.DataFrame):
  all_row = router_summary.iloc[0]
  v2_metrics_map["format_adherence"] = float(all_row["Format_Adherence_%"])
  v2_metrics_map["exact_match"] = float(all_row["Exact_Match_%"])
  v2_metrics_map["fuzzy_match"] = float(all_row["Fuzzy_Match_%"])
  if len(router_summary) > 1:
    adv_row = router_summary.iloc[1]
    v2_metrics_map["adv_format_adherence"] = float(adv_row["Format_Adherence_%"])
    v2_metrics_map["adv_exact_match"] = float(adv_row["Exact_Match_%"])
    v2_metrics_map["adv_fuzzy_match"] = float(adv_row["Fuzzy_Match_%"])
elif os.path.exists("v2_router_eval.csv"):
  _eval = pd.read_csv("v2_router_eval.csv")
  v2_metrics_map["format_adherence"] = float(_eval["format_adherence"].mean() * 100)
  v2_metrics_map["exact_match"] = float(_eval["exact_match"].mean() * 100)
  v2_metrics_map["fuzzy_match"] = float(_eval["fuzzy_match"].mean() * 100)
  if "is_adversarial" in _eval.columns and _eval["is_adversarial"].sum() > 0:
    _adv = _eval[_eval["is_adversarial"]]
    v2_metrics_map["adv_format_adherence"] = float(_adv["format_adherence"].mean() * 100)
    v2_metrics_map["adv_exact_match"] = float(_adv["exact_match"].mean() * 100)
    v2_metrics_map["adv_fuzzy_match"] = float(_adv["fuzzy_match"].mean() * 100)

if "v2_synth_summary" in globals() and isinstance(v2_synth_summary, pd.DataFrame):
  s_row = v2_synth_summary.iloc[0]
  v2_metrics_map["rouge1"] = float(s_row["ROUGE-1"])
  v2_metrics_map["rouge2"] = float(s_row["ROUGE-2"])
  v2_metrics_map["rougeL"] = float(s_row["ROUGE-L"])
  v2_metrics_map["bleu"] = float(s_row["BLEU"])
elif os.path.exists("v2_synthesis_eval.csv"):
  _syn = pd.read_csv("v2_synthesis_eval.csv")
  v2_metrics_map["rouge1"] = float(_syn["rouge1"].mean())
  v2_metrics_map["rouge2"] = float(_syn["rouge2"].mean())
  v2_metrics_map["rougeL"] = float(_syn["rougeL"].mean())
  v2_metrics_map["bleu"] = float(_syn["bleu"].mean())

# ---------------------------------------------------------
# 2. Align Metric Names & Compute Relative Delta: (v2 - v1) / v1 * 100
# ---------------------------------------------------------
def normalize_key(name: str) -> str:
  return (
      str(name)
      .lower()
      .replace("%", "")
      .replace("-", "")
      .replace(" ", "_")
      .strip("_")
  )


# Extract V1 numeric metrics regardless of whether v1_metrics.csv is wide (1-row) or long (metric/value columns)
v1_numeric = {}
cols_lower = [c.lower() for c in v1_df.columns]

if "metric" in cols_lower and any(
    k in cols_lower for k in ["value", "score", "v1", "baseline"]
):
  m_col = v1_df.columns[cols_lower.index("metric")]
  v_col = next(
      v1_df.columns[i]
      for i, c in enumerate(cols_lower)
      if c in ["value", "score", "v1", "baseline"]
  )
  for _, row in v1_df.iterrows():
    if pd.api.types.is_numeric_dtype(type(row[v_col])) or isinstance(
        row[v_col], (int, float, np.number)
    ):
      v1_numeric[normalize_key(row[m_col])] = float(row[v_col])
else:
  # Wide table: take first numeric row (or mean across rows)
  num_df = v1_df.select_dtypes(include=[np.number])
  for col in num_df.columns:
    v1_numeric[normalize_key(col)] = float(num_df[col].iloc[0])

# Map common V1 column aliases to standardized keys
ALIAS_TO_CANONICAL = {
    "format_adherence": "format_adherence",
    "json_adherence": "format_adherence",
    "json_valid": "format_adherence",
    "exact_match": "exact_match",
    "intent_accuracy": "exact_match",
    "accuracy": "exact_match",
    "fuzzy_match": "fuzzy_match",
    "fuzzy_accuracy": "fuzzy_match",
    "adv_format_adherence": "adv_format_adherence",
    "adversarial_format_adherence": "adv_format_adherence",
    "adv_exact_match": "adv_exact_match",
    "adversarial_exact_match": "adv_exact_match",
    "adversarial_accuracy": "adv_exact_match",
    "adv_fuzzy_match": "adv_fuzzy_match",
    "adversarial_fuzzy_match": "adv_fuzzy_match",
    "rouge1": "rouge1",
    "rouge_1": "rouge1",
    "rouge2": "rouge2",
    "rouge_2": "rouge2",
    "rougel": "rougeL",
    "rouge_l": "rougeL",
    "bleu": "bleu",
    "bleu_score": "bleu",
}

comparison_rows = []
for raw_k, v1_val in v1_numeric.items():
  canon_k = ALIAS_TO_CANONICAL.get(raw_k, raw_k)
  if canon_k not in v2_metrics_map:
    continue
  v2_val = v2_metrics_map[canon_k]

  # Harmonize scale if V1 stored percentages as 0-1 while V2 stored 0-100 (or vice versa)
  if v1_val <= 1.0 and v2_val > 1.5:
    v1_val = v1_val * 100.0
  elif v1_val > 1.5 and v2_val <= 1.0:
    v2_val = v2_val * 100.0

  abs_diff = v2_val - v1_val
  pct_gain = ((v2_val - v1_val) / v1_val * 100.0) if v1_val != 0 else np.nan

  comparison_rows.append({
      "Metric": canon_k,
      "V1_Naive_RAG": round(v1_val, 4),
      "V2_Hybrid_RAG": round(v2_val, 4),
      "Absolute_Delta": round(abs_diff, 4),
      "Improvement_%": round(pct_gain, 2) if not np.isnan(pct_gain) else "N/A",
  })

impact_df = pd.DataFrame(comparison_rows)

print("\n=== SOLUTION V1 (NAIVE RAG) vs SOLUTION V2 (HYBRID RAG) ===")
print(impact_df.to_string(index=False))

# Save comparison table for reference
impact_df.to_csv("v1_vs_v2_impact.csv", index=False)

# ---------------------------------------------------------
# 3. Quantitative Attribution Summary
# ---------------------------------------------------------
print("\n=== FINE-TUNING IMPACT ATTRIBUTION ===")
for _, r in impact_df.iterrows():
  direction = "gain" if r["Absolute_Delta"] >= 0 else "drop"
  print(
      f"- {r['Metric']}: {r['V1_Naive_RAG']} -> {r['V2_Hybrid_RAG']} "
      f"({r['Absolute_Delta']:+.4f} abs | {r['Improvement_%']}% relative {direction})"
  )

print(
    "\nArchitectural Attribution:\n"
    "Because both Solution V1 and Solution V2 use the exact same Chroma SOP vector store "
    "and retrieval pipeline, the observed performance delta is isolated directly to "
    "LoRA fine-tuning of the intent router (structured JSON intent extraction -> targeted "
    "SOP header lookup instead of noisy raw user query embeddings)."
)

=== RAW v1_metrics.csv ===
 row_idx                                                                                                                                                                                                                                                                                                                                                                               query                   intent     category                                                                 baseline_output                                                                                                                                                                                                                                                                                                                                                                                                                               rag_output                                retrieved_source  

### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [10]:
# YOUR CODE HERE
import difflib
import json
import os
import re
import subprocess
import sys
import time
import pandas as pd

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "rouge-score",
    "nltk",
    "tqdm",
])

from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from rouge_score import rouge_scorer
import torch
from tqdm.auto import tqdm

# 1. Load held-out test set
df_test = pd.read_csv("df_test.csv")
query_col = next(
    (
        c
        for c in df_test.columns
        if c.lower()
        in ["query", "text", "input", "instruction", "utterance", "prompt"]
    ),
    df_test.columns[0],
)
intent_col = next(
    (
        c
        for c in df_test.columns
        if c.lower()
        in ["intent", "label", "target", "category", "output", "ground_truth"]
    ),
    df_test.columns[1] if len(df_test.columns) > 1 else df_test.columns[0],
)
ref_col = next(
    (
        c
        for c in df_test.columns
        if c.lower()
        in [
            "reference",
            "expected_answer",
            "sop_answer",
            "ground_truth_answer",
            "response",
            "answer",
        ]
    ),
    None,
)

# Set MAX_ROWS = 50 for fast T4 runs; set to None to run the entire df_test
MAX_ROWS = 50
eval_slice = (
    df_test.head(MAX_ROWS).copy()
    if MAX_ROWS is not None
    else df_test.copy()
)


def parse_intent_label(val) -> str:
  s = str(val).strip()
  m = re.search(r"\{.*?\}", s, flags=re.DOTALL)
  if m:
    try:
      obj = json.loads(m.group(0))
      if isinstance(obj, dict) and "intent" in obj:
        return str(obj["intent"]).strip().lower()
    except json.JSONDecodeError:
      pass
  return s.strip('"').strip("'").strip().lower()


eval_slice["true_intent"] = eval_slice[intent_col].apply(parse_intent_label)

# Optional reference answers from outputs.json
ref_lookup = {}
if os.path.exists("outputs.json"):
  with open("outputs.json", "r") as f:
    raw_refs = json.load(f)
  if isinstance(raw_refs, list):
    for item in raw_refs:
      if isinstance(item, dict):
        q = item.get("query") or item.get("question") or item.get("input")
        r = (
            item.get("reference")
            or item.get("ground_truth")
            or item.get("expected_answer")
            or item.get("answer")
            or item.get("output")
        )
        if q and r:
          ref_lookup[str(q).strip()] = str(r).strip()
  elif isinstance(raw_refs, dict):
    for k, v in raw_refs.items():
      if isinstance(v, str):
        ref_lookup[str(k).strip()] = v.strip()
      elif isinstance(v, dict):
        q = v.get("query") or str(k)
        r = (
            v.get("reference")
            or v.get("ground_truth")
            or v.get("answer")
            or v.get("output")
        )
        if r:
          ref_lookup[str(q).strip()] = str(r).strip()

assert (
    "router_model" in globals()
    and "tokenizer" in globals()
    and "collection" in globals()
), "Please run Task 4.3.1 first."

INTENT_TO_SOP_QUERY.setdefault("check_delivery_status", "# Shipping Delays")


def run_llm(prompt_text: str, max_new_tokens: int = 160) -> str:
  inputs = tokenizer(prompt_text, return_tensors="pt").to(router_model.device)
  with torch.no_grad():
    outputs = router_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
  gen_tokens = outputs[0][inputs["input_ids"].shape[1] :]
  return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


def format_chat(system_msg: str, user_msg: str) -> str:
  msgs = [
      {"role": "system", "content": system_msg},
      {"role": "user", "content": user_msg},
  ]
  if hasattr(tokenizer, "apply_chat_template"):
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
  return f"{system_msg}\nUser: {user_msg}\nAssistant:"


def baseline_answer(query: str) -> str:
  prompt = format_chat(
      "You are a helpful customer-support assistant. Answer concisely and accurately.",
      query,
  )
  return run_llm(prompt, max_new_tokens=160)


def naive_rag_answer(query: str, k: int = 3) -> tuple[str, list[str]]:
  docs = retrieve_sop(query, k=k)
  ctx = "\n\n---\n\n".join(docs) if docs else "No SOP context available."
  prompt = format_chat(
      "You are a customer-support assistant. Answer ONLY using the provided SOP context.",
      f"SOP Context:\n{ctx}\n\nCustomer Query: {query}",
  )
  return run_llm(prompt, max_new_tokens=160), docs


def sop_grounded_reference(
    query: str, true_intent: str, sop_docs: list[str]
) -> str:
  if query.strip() in ref_lookup:
    return ref_lookup[query.strip()]
  if not sop_docs:
    sop_docs = retrieve_sop(
        intent_to_search_string(true_intent, query, True), k=3
    )
  ctx = "\n\n---\n\n".join(sop_docs) if sop_docs else "General support policy."
  prompt = format_chat(
      "Create a concise, accurate SOP-grounded customer-support reference answer strictly supported by the SOP text.",
      f"Intent: {true_intent}\nSOP Policy:\n{ctx}\n\nCustomer Query: {query}",
  )
  return run_llm(prompt, max_new_tokens=160)


scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1


def score_text(ref: str, pred: str) -> dict:
  if not pred.strip():
    return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0, "bleu": 0.0}
  sc = scorer.score(ref, pred)
  ref_toks = ref.lower().split()
  pred_toks = pred.lower().split()
  bleu_val = (
      sentence_bleu([ref_toks], pred_toks, smoothing_function=smooth)
      if ref_toks and pred_toks
      else 0.0
  )
  return {
      "rouge1": sc["rouge1"].fmeasure,
      "rouge2": sc["rouge2"].fmeasure,
      "rougeL": sc["rougeL"].fmeasure,
      "bleu": bleu_val,
  }


rows = []
for idx, row in tqdm(
    eval_slice.iterrows(),
    total=len(eval_slice),
    desc="Part 1: Running Baseline, Naive RAG, Hybrid RAG",
):
  q = str(row[query_col])
  t_intent = str(row["true_intent"])
  row_explicit_ref = (
      str(row[ref_col]).strip()
      if (ref_col and pd.notna(row[ref_col]) and str(row[ref_col]).strip())
      else None
  )

  # 1. Baseline
  t0 = time.perf_counter()
  base_ans = baseline_answer(q)
  base_lat = time.perf_counter() - t0

  # 2. Naive RAG
  t0 = time.perf_counter()
  naive_ans, naive_docs = naive_rag_answer(q, k=3)
  naive_lat = time.perf_counter() - t0

  # 3. Hybrid RAG
  t0 = time.perf_counter()
  route_info = route_intent(q)
  pred_intent = route_info["intent"].strip().lower()
  search_str = intent_to_search_string(
      pred_intent, q, route_info["parsed_ok"]
  )
  hybrid_docs = retrieve_sop(search_str, k=3)
  hybrid_ans = generate_answer(q, hybrid_docs)
  hybrid_lat = time.perf_counter() - t0

  ref_ans = (
      row_explicit_ref
      or ref_lookup.get(q.strip())
      or sop_grounded_reference(q, t_intent, hybrid_docs or naive_docs)
  )

  fmt_ok = bool(route_info["parsed_ok"])
  exact_ok = pred_intent == t_intent
  fuzzy_ratio = difflib.SequenceMatcher(None, pred_intent, t_intent).ratio()
  fuzzy_ok = fuzzy_ratio >= 0.80

  b_sc = score_text(ref_ans, base_ans)
  n_sc = score_text(ref_ans, naive_ans)
  h_sc = score_text(ref_ans, hybrid_ans)

  rows.append({
      "row_id": idx,
      "query": q,
      "true_intent": t_intent,
      "pred_intent": pred_intent,
      "router_format_adherent": fmt_ok,
      "router_exact_match": exact_ok,
      "router_fuzzy_match": fuzzy_ok,
      "router_fuzzy_similarity": round(fuzzy_ratio, 4),
      "hybrid_search_query": search_str,
      "sop_reference": ref_ans,
      "baseline_answer": base_ans,
      "baseline_rouge1": round(b_sc["rouge1"], 4),
      "baseline_rouge2": round(b_sc["rouge2"], 4),
      "baseline_rougeL": round(b_sc["rougeL"], 4),
      "baseline_bleu": round(b_sc["bleu"], 4),
      "baseline_latency_s": round(base_lat, 3),
      "naive_rag_answer": naive_ans,
      "naive_rag_rouge1": round(n_sc["rouge1"], 4),
      "naive_rag_rouge2": round(n_sc["rouge2"], 4),
      "naive_rag_rougeL": round(n_sc["rougeL"], 4),
      "naive_rag_bleu": round(n_sc["bleu"], 4),
      "naive_rag_latency_s": round(naive_lat, 3),
      "hybrid_rag_answer": hybrid_ans,
      "hybrid_rag_rouge1": round(h_sc["rouge1"], 4),
      "hybrid_rag_rouge2": round(h_sc["rouge2"], 4),
      "hybrid_rag_rougeL": round(h_sc["rougeL"], 4),
      "hybrid_rag_bleu": round(h_sc["bleu"], 4),
      "hybrid_rag_latency_s": round(hybrid_lat, 3),
  })

full_df = pd.DataFrame(rows)
full_df.to_csv("Comparative_Results_Full.csv", index=False)
print("Saved Comparative_Results_Full.csv:", full_df.shape)

Part 1: Running Baseline, Naive RAG, Hybrid RAG:   0%|          | 0/50 [00:00<?, ?it/s]

Saved Comparative_Results_Full.csv: (50, 28)


In [11]:
# YOUR CODE HERE
import os
import numpy as np
import pandas as pd

if "full_df" not in globals() or not isinstance(full_df, pd.DataFrame):
  assert os.path.exists(
      "Comparative_Results_Full.csv"
  ), "Run Part 1 first to generate Comparative_Results_Full.csv."
  full_df = pd.read_csv("Comparative_Results_Full.csv")

fmt_pct = round(full_df["router_format_adherent"].mean() * 100, 2)
exact_pct = round(full_df["router_exact_match"].mean() * 100, 2)
fuzzy_pct = round(full_df["router_fuzzy_match"].mean() * 100, 2)

summary_rows = [
    {
        "Architecture": "Baseline (LLM Only)",
        "Samples": len(full_df),
        "Router_Format_Adherence_%": np.nan,
        "Router_Exact_Match_%": np.nan,
        "Router_Fuzzy_Match_%": np.nan,
        "ROUGE-1": round(full_df["baseline_rouge1"].mean(), 4),
        "ROUGE-2": round(full_df["baseline_rouge2"].mean(), 4),
        "ROUGE-L": round(full_df["baseline_rougeL"].mean(), 4),
        "BLEU": round(full_df["baseline_bleu"].mean(), 4),
        "Avg_Latency_s": round(full_df["baseline_latency_s"].mean(), 3),
    },
    {
        "Architecture": "Naive RAG (Vector Query)",
        "Samples": len(full_df),
        "Router_Format_Adherence_%": np.nan,
        "Router_Exact_Match_%": np.nan,
        "Router_Fuzzy_Match_%": np.nan,
        "ROUGE-1": round(full_df["naive_rag_rouge1"].mean(), 4),
        "ROUGE-2": round(full_df["naive_rag_rouge2"].mean(), 4),
        "ROUGE-L": round(full_df["naive_rag_rougeL"].mean(), 4),
        "BLEU": round(full_df["naive_rag_bleu"].mean(), 4),
        "Avg_Latency_s": round(full_df["naive_rag_latency_s"].mean(), 3),
    },
    {
        "Architecture": "Hybrid RAG (Fine-Tuned Router + SOP Retrieval)",
        "Samples": len(full_df),
        "Router_Format_Adherence_%": fmt_pct,
        "Router_Exact_Match_%": exact_pct,
        "Router_Fuzzy_Match_%": fuzzy_pct,
        "ROUGE-1": round(full_df["hybrid_rag_rouge1"].mean(), 4),
        "ROUGE-2": round(full_df["hybrid_rag_rouge2"].mean(), 4),
        "ROUGE-L": round(full_df["hybrid_rag_rougeL"].mean(), 4),
        "BLEU": round(full_df["hybrid_rag_bleu"].mean(), 4),
        "Avg_Latency_s": round(full_df["hybrid_rag_latency_s"].mean(), 3),
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("Comparative_Results_Summary.csv", index=False)

print("Saved Comparative_Results_Summary.csv:", summary_df.shape)
print("\n=== COMPARATIVE RESULTS SUMMARY ===")
print(summary_df.to_string(index=False))

Saved Comparative_Results_Summary.csv: (3, 10)

=== COMPARATIVE RESULTS SUMMARY ===
                                  Architecture  Samples  Router_Format_Adherence_%  Router_Exact_Match_%  Router_Fuzzy_Match_%  ROUGE-1  ROUGE-2  ROUGE-L   BLEU  Avg_Latency_s
                           Baseline (LLM Only)       50                        NaN                   NaN                   NaN   0.3398   0.1367   0.2748 0.0597          0.894
                      Naive RAG (Vector Query)       50                        NaN                   NaN                   NaN   0.4502   0.2576   0.3776 0.1718          1.694
Hybrid RAG (Fine-Tuned Router + SOP Retrieval)       50                      100.0                  28.0                  32.0   0.2301   0.0634   0.1479 0.0248          4.679


---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**